In [1]:
import polars as pl
import numpy as np
import time
from gensim.corpora import Dictionary
from gensim.models import LdaMulticore
from gensim.parsing.preprocessing import STOPWORDS
import joblib
from tqdm import tqdm
from bertopic import BERTopic
import os

/home/javclamar/Projects/tfg-sentiment-analysis/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
import polars as pl
import time
import os
from tqdm import tqdm
from gensim.corpora import Dictionary
from gensim.models import LdaMulticore
from gensim.parsing.preprocessing import STOPWORDS

csv_reviews = '../data/csv/yelp_academic_dataset_review.csv'
total_rows = 6_990_280
batch_size = 50000
expected_batches = total_rows // batch_size + (1 if total_rows % batch_size != 0 else 0)

stopwords_list = list(STOPWORDS) + [""]

def lda_model_training(input_csv):
    os.makedirs('../data/models/lda', exist_ok=True)

    sample_df = (
        pl.scan_csv(input_csv)
        .select('text')
        .collect()
        .sample(n=300000, seed=42) 
    )
    
    sample_df = sample_df.with_columns(
        pl.col('text')
        .str.replace_all(r'[^a-zA-Z\s]', '')
        .str.to_lowercase()
        .str.split(" ")
        .list.set_difference(stopwords_list)
        .alias('tokens')
    )

    tokenized_docs = sample_df['tokens'].to_list()
    del sample_df
    
    dictionary = Dictionary(tokenized_docs)
    dictionary.filter_extremes(no_below=10, no_above=0.85, keep_n=25000)
    del tokenized_docs

    workers_count = max(1, os.cpu_count() - 1)
    chunk_size = max(1, batch_size // workers_count)
    
    lda = LdaMulticore(
        num_topics=15,             
        id2word=dictionary,
        workers=workers_count,
        chunksize=chunk_size,
        random_state=42
    )

    start_time = time.time()
    reader = pl.read_csv_batched(input_csv, batch_size=batch_size)

    with tqdm(total=expected_batches, desc="Training LDA", unit="batch") as pbar:
        while True:
            batches = reader.next_batches(1)
            if not batches:
                break
            
            # Preprocesamiento
            chunk = batches[0].with_columns(
                pl.col('text')
                .str.replace_all(r'[^a-zA-Z\s]', '')
                .str.to_lowercase()
                .str.split(" ")
                .list.set_difference(stopwords_list)
                .alias('tokens')
            )
            
            corpus_chunk = [dictionary.doc2bow(tokens) for tokens in chunk['tokens'].to_list()]
            
            # Entrenamiento
            lda.update(corpus_chunk)
            pbar.update(1)
            
    lda.save('../data/models/lda/yelp_lda_model.gensim')
    dictionary.save('../data/models/lda/yelp_dictionary.dict')
    
    print(f"Entrenamiento completado en {(time.time() - start_time) / 60:.2f} minutos.")

lda_model_training(csv_reviews)

Training LDA: 100%|███████████████████████████████████████████████████████████████████████| 140/140 [15:23<00:00,  6.60s/batch]

Entrenamiento completado en 15.40 minutos.


In [4]:
csv_reviews = '../data/csv/yelp_academic_dataset_review.csv'

def bertopic_model_training(input_csv):
    os.makedirs('../data/models/bertopic', exist_ok=True)
    
    # Sample para entrenar BERTopic
    sample_df = (
        pl.scan_csv(input_csv)
        .select('text')
        .collect()
        .sample(n=300000, seed=42) 
    )
    
    docs = sample_df['text'].to_list()
    del sample_df
    
    start_time = time.time()

    # Entrenamiento de BERTopic
    topic_model = BERTopic(language="english", calculate_probabilities=False, verbose=True)
    topics, probs = topic_model.fit_transform(docs)
    
    topic_model.save('../data/models/bertopic/yelp_bertopic_model', serialization="safetensors", save_ctfidf=True)
    
    print(f"Entrenamiento completado en {(time.time() - start_time) / 60:.2f} minutes.")

bertopic_model_training(csv_reviews)

2026-04-07 18:40:55,675 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|█████████████████████████████| 103/103 [00:00<00:00, 725.40it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|█████████████████████████████████████████████████████████████████████████████| 9375/9375 [04:34<00:00, 34.21it/s]
2026-04-07 18:45:40,539 - BERTopic - Embedding - Completed ✓
2026-04-07 18:45:40,540 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-07 18:49:11,213 - BERTopic - Dimensionality - Completed ✓
2026-04-07 18:49:11,231 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-07 18:49:41,203 -

Entrenamiento completado en 9.29 minutes.


In [2]:
csv_reviews = '../data/csv/yelp_academic_dataset_review.csv'
csv_reviews_output_topics = '../results/topic_modeling/yelp_academic_dataset_review_topics.csv'
lda_model_path = '../data/models/lda/yelp_lda_model.gensim'
dictionary_path = '../data/models/lda/yelp_dictionary.dict'
bertopic_model_path = '../data/models/bertopic/yelp_bertopic_model'
total_rows = 6_990_280
batch_size = 50000
expected_batches = total_rows // batch_size + (1 if total_rows % batch_size != 0 else 0)

stopwords_list = list(STOPWORDS) + [""]

def topic_modeling(input_csv, output_csv, lda_model_path='../data/models/lda/yelp_lda_model.gensim', 
                             dictionary_path='../data/models/lda/yelp_dictionary.dict',
                             bertopic_model_path='../data/models/bertopic/yelp_bertopic_model'):

    lda = LdaMulticore.load(lda_model_path)
    dictionary = Dictionary.load(dictionary_path)
    
    bertopic_model = BERTopic.load(bertopic_model_path)
    bertopic_model.verbose = False

    lda_topic_words = {}
    for topic_id in range(lda.num_topics):
        words = [dictionary[word_id] for word_id, _ in lda.get_topic_terms(topic_id, topn=3)]
        lda_topic_words[topic_id] = ", ".join(words)
    
    bertopic_labels_dict = {-1: 'outlier'}
    for topic_id in bertopic_model.get_topic_info()['Topic']:
        if topic_id != -1:
            rep = bertopic_model.get_topic(topic_id)
            if rep:
                bertopic_labels_dict[topic_id] = ', '.join([word for word, _ in rep[:3]])
            else:
                bertopic_labels_dict[topic_id] = f'topic_{topic_id}'

    start_time = time.time()

    total_rows = 6_990_280
    batch_size = 50000  
    expected_batches = total_rows // batch_size + (1 if total_rows % batch_size != 0 else 0)
    
    reader = pl.read_csv_batched(input_csv, batch_size=batch_size)
    batch_count = 0

    with tqdm(total=expected_batches, desc="Topic Modeling", unit="batch") as pbar:
        while True:
            batches = reader.next_batches(1)
            if not batches:
                break
            
            chunk = batches[0]
            
            chunk_lda = chunk.with_columns(
                pl.col('text')
                .str.replace_all(r'[^a-zA-Z\s]', '')
                .str.to_lowercase()
                .str.split(" ")
                .list.set_difference(stopwords_list)
                .alias('tokens')
            )
            
            bow_corpus = [dictionary.doc2bow(tokens) for tokens in chunk_lda['tokens'].to_list()]
            topic_distributions = lda[bow_corpus]
            
            dominant_topics = []
            topic_probs = []
            
            for dist in topic_distributions:
                if dist:
                    best_topic_id, best_prob = max(dist, key=lambda x: x[1])
                    dominant_topics.append(lda_topic_words[best_topic_id])
                    topic_probs.append(best_prob)
                else:
                    dominant_topics.append("none")
                    topic_probs.append(0.0)
            

            raw_texts = chunk['text'].to_list()
            bertopic_topics, _ = bertopic_model.transform(raw_texts)
            bertopic_topic_labels = [bertopic_labels_dict.get(t, f'topic_{t}') for t in bertopic_topics]
            
            chunk_final = chunk.with_columns([
                pl.Series('lda_dominant_topic', dominant_topics),
                pl.Series('lda_topic_probability', topic_probs),
                pl.Series('bertopic_topic', bertopic_topics),
                pl.Series('bertopic_dominant_topic', bertopic_topic_labels)
            ])
            
            mode = 'wb' if batch_count == 0 else 'ab'
            with open(output_csv, mode) as f:
                chunk_final.write_csv(f, include_header=(batch_count == 0))
    
            batch_count += 1
            pbar.update(1)
    
    print(f"Resultados guardados dentro de {output_csv} en {(time.time() - start_time) / 60:.2f} minutes.")

topic_modeling(csv_reviews, csv_reviews_output_topics, lda_model_path, dictionary_path, bertopic_model_path)

Loading weights: 100%|█████████████████████████████| 103/103 [00:00<00:00, 817.81it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Topic Modeling: 100%|███████████████████████████████████████████████████████████████████| 140/140 [2:22:27<00:00, 61.05s/batch]

Resultados guardados dentro de ../results/topic_modeling/yelp_academic_dataset_review_topics.csv en 142.45 minutes.


In [3]:
csv_reviews_output_topics = '../results/topic_modeling/yelp_academic_dataset_review_topics.csv'

print(pl.scan_csv(csv_reviews_output_topics, ignore_errors=True)
    .tail(5).collect())

shape: (5, 11)
┌────────────┬────────────┬────────────┬───────┬───┬───────────┬───────────┬───────────┬───────────┐
│ review_id  ┆ user_id    ┆ business_i ┆ stars ┆ … ┆ lda_domin ┆ lda_topic ┆ bertopic_ ┆ bertopic_ │
│ ---        ┆ ---        ┆ d          ┆ ---   ┆   ┆ ant_topic ┆ _probabil ┆ topic     ┆ dominant_ │
│ str        ┆ str        ┆ ---        ┆ f64   ┆   ┆ ---       ┆ ity       ┆ ---       ┆ topic     │
│            ┆            ┆ str        ┆       ┆   ┆ str       ┆ ---       ┆ i64       ┆ ---       │
│            ┆            ┆            ┆       ┆   ┆           ┆ f64       ┆           ┆ str       │
╞════════════╪════════════╪════════════╪═══════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ H0RIamZu0B ┆ qskILQ3k0I ┆ jals67o91g ┆ 5.0   ┆ … ┆ time,     ┆ 0.581336  ┆ 601       ┆ card,     │
│ 0Ei0P4aeh3 ┆ _qcCMI-k6_ ┆ crD4DC81Vk ┆       ┆   ┆ work,     ┆           ┆           ┆ debit,    │
│ sQ         ┆ QQ         ┆ 6w         ┆       ┆   ┆ service   ┆           ┆